# MobileNetV2 - Face Recognition - Metric Learning

### Google Colab Setup
1. **Runtime > Change runtime type > T4 GPU.**
2. Join the competition on Kaggle once (open its page and accept the rules), otherwise the download fails with a 403.
3. Kaggle login: add Colab Secrets named `KAGGLE_USERNAME` and `KAGGLE_KEY` (key icon in the left sidebar, enable notebook access), or just enter your credentials when prompted.
4. Run the cells top to bottom.

**If the session ends before training finishes:** open a new session and run all cells again. The dataset is downloaded again (Colab's local disk is wiped), and the training cell automatically resumes from the last checkpoint saved on Google Drive.

In [2]:
CA# ==========================================
# Google Colab setup: packages, Drive, dataset
# Run this cell first in every new session.
# ==========================================

import os
import sys
import time
import zipfile
from pathlib import Path

SESSION_START = time.time()

# 1. Packages
%pip install -q -U kagglehub timm

# 2. Google Drive: checkpoints are saved here so they survive the end of a session
from google.colab import drive
drive.mount("/content/drive")

CHECKPOINT_DIR = "/content/drive/MyDrive/face_recognition/models"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 3. Kaggle credentials: Colab Secrets if set, otherwise an interactive login prompt
os.environ["KAGGLEHUB_CACHE"] = "/content/kaggle_cache"  # Colab's local disk: fast reads (not Drive)

import kagglehub

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception:
    kagglehub.login()

# 4. Download the dataset to local disk (cached for the rest of this session)
download_path = Path(kagglehub.competition_download("11-785-fall-20-homework-2-part-2"))
print("Path to competition files:", download_path)


def find_in(root, name, max_depth=3):
    """Find `name` at most `max_depth` folder levels below root (avoids walking every image)."""
    for level in range(max_depth):
        matches = list(root.glob("/".join(["*"] * level + [name])))
        if matches:
            return matches[0]
    return None


# Extract any zip archives that were not unpacked automatically
if find_in(download_path, "classification_data") is None:
    for archive in download_path.glob("*.zip"):
        print(f"Extracting {archive.name} ...")
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(download_path)

classification_dir = find_in(download_path, "classification_data")
pairs_file = find_in(download_path, "verification_pairs_val.txt")

if classification_dir is None or pairs_file is None:
    raise FileNotFoundError(
        "Expected 'classification_data/' and 'verification_pairs_val.txt' inside the download. "
        f"Found: {sorted(p.name for p in download_path.iterdir())}"
    )

DATA_ROOT = pairs_file.parent  # folder that the image paths in the pairs file are relative to
TRAIN_DIR = str(classification_dir / "train_data")
VAL_DIR = str(classification_dir / "val_data")
VAL_PAIRS_PATH = str(pairs_file)

for required in (TRAIN_DIR, VAL_DIR):
    if not os.path.isdir(required):
        raise FileNotFoundError(f"Missing folder: {required}")

print("Data root      :", DATA_ROOT)
print("Train folder   :", TRAIN_DIR)
print("Val pairs file :", VAL_PAIRS_PATH)
print("Checkpoints in :", CHECKPOINT_DIR)
print(f"Setup finished in {(time.time() - SESSION_START) / 60:.1f} min")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


100%|██████████| 1.35G/1.35G [00:19<00:00, 72.9MB/s]

Extracting files...


Path to competition files: /content/kaggle_cache/competitions/11-785-fall-20-homework-2-part-2
Data root      : /content/kaggle_cache/competitions/11-785-fall-20-homework-2-part-2
Train folder   : /content/kaggle_cache/competitions/11-785-fall-20-homework-2-part-2/classification_data/train_data
Val pairs file : /content/kaggle_cache/competitions/11-785-fall-20-homework-2-part-2/verification_pairs_val.txt
Checkpoints in : /content/drive/MyDrive/face_recognition/models
Setup finished in 1.6 min
Kaggle credentials set.
Kaggle credentials successfully validated.


### Import Libraries

In [3]:
# Standard libraries
import os
import time
from pathlib import Path

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.amp import GradScaler, autocast

# TorchVision
from torchvision import datasets, transforms

# Data processing
import numpy as np
import pandas as pd
from PIL import Image

# Evaluation metrics
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    roc_curve
)

# Progress bar
from tqdm.auto import tqdm

# Model
import timm


# Device configuration (Colab T4 GPU)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU.")

device = torch.device("cuda")
torch.backends.cudnn.benchmark = True  # fixed 112x112 input size, so cuDNN autotuning pays off
NUM_WORKERS = os.cpu_count() or 2      # Colab typically has only 2 CPU cores
print(f"Using device: {device} ({torch.cuda.get_device_name(0)})")

Using device: cuda (Tesla T4)


### Load Dataset

In [4]:
# TRAIN_DIR, VAL_DIR, VAL_PAIRS_PATH and DATA_ROOT are set in the Colab setup cell above

# Custom Dataset class to load verification pairs for face verification evaluation
class VerificationDataset(Dataset):
    def __init__(self, pairs_file, root_dir, transform=None):
        self.df = pd.read_csv(
            pairs_file, sep=r'\s+', header=None,
            names=['img1', 'img2', 'label']
        )
        self.root_dir = Path(root_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = row['label']

        img1 = Image.open(self.root_dir / row['img1']).convert('RGB')
        img2 = Image.open(self.root_dir / row['img2']).convert('RGB')

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, torch.tensor(float(label) if not pd.isna(label) else -1.0)

print("Dataset paths and verification dataset class defined successfully.")


Dataset paths and verification dataset class defined successfully.


### Preprocessing

In [5]:
# Define image augmentations and preprocessing pipelines
train_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

eval_transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Load training classification dataset (ImageFolder structure maps subfolder names as class IDs)
train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transform)
train_loader = DataLoader(
    train_dataset, batch_size=128, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

# Load validation verification pairs dataset
val_verification_dataset = VerificationDataset(
    pairs_file=VAL_PAIRS_PATH, root_dir=DATA_ROOT, transform=eval_transform
)
val_verification_loader = DataLoader(
    val_verification_dataset, batch_size=256, shuffle=False, num_workers=NUM_WORKERS
)

print(f"Loaded {len(train_dataset.classes)} training classes with {len(train_dataset)} images.")
print(f"Loaded {len(val_verification_dataset)} validation verification pairs.")

Loaded 4000 training classes with 380638 images.
Loaded 8805 validation verification pairs.


### Train Model

In [6]:
# ==========================================
# 1. Device configuration (set in the import cell)
# ==========================================

print(f"Using device: {device}")


# ==========================================
# 2. Define ArcFace Metric Learning Head
# ==========================================

class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50):
        super(ArcMarginProduct, self).__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m

        self.weight = nn.Parameter(
            torch.FloatTensor(out_features, in_features)
        )

        nn.init.xavier_uniform_(self.weight)

    def forward(self, input, label):
        # Run the ArcFace maths in float32 (stable under mixed precision)
        input = input.float()

        with torch.autocast(device_type="cuda", enabled=False):
            cosine = F.linear(
                F.normalize(input),
                F.normalize(self.weight)
            )

            sine = torch.sqrt(
                1.0 - torch.pow(cosine, 2).clamp(0, 1)
            )

            phi = (
                cosine * np.cos(self.m)
                - sine * np.sin(self.m)
            )

            one_hot = torch.zeros(
                cosine.size(),
                device=input.device
            )

            one_hot.scatter_(
                1,
                label.view(-1, 1).long(),
                1
            )

            output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
            output *= self.s

        return output


# ==========================================
# 3. Define MobileNetV2 Face Model
#    Partial Fine-Tuning
# ==========================================

class FaceModel(nn.Module):
    def __init__(self, num_classes, embedding_size=512):
        super(FaceModel, self).__init__()

        # Load ImageNet-pretrained MobileNetV2 (Global Average Pooling, no classifier)
        self.backbone = timm.create_model(
            'mobilenetv2_100',
            pretrained=True,
            num_classes=0,
            global_pool='avg'
        )

        # Freeze all backbone parameters
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Unfreeze only the final (stride-32) stage of MobileNetV2:
        # inverted-residual blocks 5 and 6, plus the 1x1 head conv and its BN.
        # This is the MobileNetV2 equivalent of ResNet34's layer4.
        for module in (
            self.backbone.blocks[5],
            self.backbone.blocks[6],
            self.backbone.conv_head,
            self.backbone.bn2,
        ):
            for param in module.parameters():
                param.requires_grad = True

        in_features = self.backbone.num_features

        # Embedding layer
        self.embedding_layer = nn.Linear(
            in_features,
            embedding_size
        )

        self.bn = nn.BatchNorm1d(embedding_size)

        # ArcFace classification head
        self.head = ArcMarginProduct(
            embedding_size,
            num_classes
        )

    # Keep frozen BatchNorm layers in evaluation mode
    def train(self, mode=True):
        super().train(mode)

        for module in self.backbone.modules():
            if isinstance(module, nn.BatchNorm2d):
                if not any(param.requires_grad for param in module.parameters()):
                    module.eval()

        return self

    def forward(self, x, label=None):
        features = self.backbone(x)

        embeddings = self.embedding_layer(features)
        embeddings = self.bn(embeddings)

        if label is not None:
            logits = self.head(embeddings, label)
            return embeddings, logits

        return F.normalize(embeddings, p=2, dim=1)


# ==========================================
# 4. Initialize Model
# ==========================================

model = FaceModel(
    num_classes=len(train_dataset.classes),
    embedding_size=512
).to(device)


# ==========================================
# 5. Display Trainable and Frozen Layers
# ==========================================

print("\nTrainable parameters:")

for name, param in model.named_parameters():
    if param.requires_grad:
        print(name)

print("\nFrozen parameters:")

for name, param in model.named_parameters():
    if not param.requires_grad:
        print(name)


# ==========================================
# 6. Initialize Optimizer and Loss
# ==========================================

# Optimize only trainable parameters
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss()


# ==========================================
# 7. Checkpoint Configuration
# ==========================================

# Checkpoints live on Google Drive so they survive the end of a Colab session
# (CHECKPOINT_DIR is created in the Colab setup cell)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "mobilenetv2_metric_learning_checkpoint.pth")

scaler = GradScaler("cuda")  # mixed-precision loss scaling

start_epoch = 0
epochs = 20

def save_checkpoint(checkpoint, path):
    # Write to a temp file first, so a session that dies mid-save cannot corrupt the checkpoint
    tmp_path = path + ".tmp"
    torch.save(checkpoint, tmp_path)
    os.replace(tmp_path, path)


# ==========================================
# 8. Automatically Resume Training
# ==========================================

if os.path.exists(CHECKPOINT_PATH):

    print(f"\nFound existing checkpoint: {CHECKPOINT_PATH}")
    print("Resuming training...")

    # Load checkpoint onto CPU first, then copy it into the model on the GPU
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False
    )

    model.load_state_dict(checkpoint['model_state_dict'])

    optimizer.load_state_dict(
        checkpoint['optimizer_state_dict']
    )

    if 'scaler_state_dict' in checkpoint:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

    start_epoch = checkpoint['epoch'] + 1

    print(f"Resuming from Epoch {start_epoch + 1}/{epochs}")

else:

    print("\nNo checkpoint found.")
    print("Starting training from scratch.")


# ==========================================
# 9. Training Loop
# ==========================================

model.train()

for epoch in range(start_epoch, epochs):

    epoch_start = time.time()

    loop = tqdm(
        train_loader,
        desc=f"Epoch [{epoch + 1}/{epochs}]"
    )

    for images, labels in loop:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        # Forward pass + ArcFace loss (mixed precision)
        with autocast("cuda"):
            _, logits = model(images, labels)
            loss = criterion(logits, labels)

        # Backpropagation
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loop.set_postfix(loss=loss.item())


    # ======================================
    # 10. Save Checkpoint
    # ======================================

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict()
    }

    save_checkpoint(checkpoint, CHECKPOINT_PATH)

    print(
        f"Checkpoint successfully saved for "
        f"Epoch {epoch + 1} at {CHECKPOINT_PATH} "
        f"(epoch took {(time.time() - epoch_start) / 60:.1f} min)"
    )

print("\nTraining completed!")

Using device: cuda


model.safetensors: reconstructing file:   0%|          |  0.00B / 14.2MB            

model.safetensors: downloading bytes:           |  0.00B            


Trainable parameters:
backbone.blocks.5.0.conv_pw.weight
backbone.blocks.5.0.bn1.weight
backbone.blocks.5.0.bn1.bias
backbone.blocks.5.0.conv_dw.weight
backbone.blocks.5.0.bn2.weight
backbone.blocks.5.0.bn2.bias
backbone.blocks.5.0.conv_pwl.weight
backbone.blocks.5.0.bn3.weight
backbone.blocks.5.0.bn3.bias
backbone.blocks.5.1.conv_pw.weight
backbone.blocks.5.1.bn1.weight
backbone.blocks.5.1.bn1.bias
backbone.blocks.5.1.conv_dw.weight
backbone.blocks.5.1.bn2.weight
backbone.blocks.5.1.bn2.bias
backbone.blocks.5.1.conv_pwl.weight
backbone.blocks.5.1.bn3.weight
backbone.blocks.5.1.bn3.bias
backbone.blocks.5.2.conv_pw.weight
backbone.blocks.5.2.bn1.weight
backbone.blocks.5.2.bn1.bias
backbone.blocks.5.2.conv_dw.weight
backbone.blocks.5.2.bn2.weight
backbone.blocks.5.2.bn2.bias
backbone.blocks.5.2.conv_pwl.weight
backbone.blocks.5.2.bn3.weight
backbone.blocks.5.2.bn3.bias
backbone.blocks.6.0.conv_pw.weight
backbone.blocks.6.0.bn1.weight
backbone.blocks.6.0.bn1.bias
backbone.blocks.6.0.conv

Epoch [1/20]:   0%|          | 0/2974 [00:00<?, ?it/s]

Checkpoint successfully saved for Epoch 1 at /content/drive/MyDrive/face_recognition/models/mobilenetv2_metric_learning_checkpoint.pth (epoch took 11.0 min)


... (34 outputs left out)

Epoch [19/20]:   0%|          | 0/2974 [00:00<?, ?it/s]

Checkpoint successfully saved for Epoch 19 at /content/drive/MyDrive/face_recognition/models/mobilenetv2_metric_learning_checkpoint.pth (epoch took 10.6 min)


Epoch [20/20]:   0%|          | 0/2974 [00:00<?, ?it/s]

Checkpoint successfully saved for Epoch 20 at /content/drive/MyDrive/face_recognition/models/mobilenetv2_metric_learning_checkpoint.pth (epoch took 10.8 min)

Training completed!


### Evaluate Model

In [7]:

# ==========================================
# MOBILENETV2 FACE VERIFICATION VALIDATION
# Supports Classification and Metric Learning
# ==========================================

from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# 1. Configuration
MODEL_TYPE = "metric_learning"  # "classification" or "metric_learning"

CURRENT_DIR = Path.cwd()
CHECKPOINT_NAME = f"mobilenetv2_{MODEL_TYPE}_checkpoint.pth"

# Checkpoint comes from Google Drive; VAL_PAIRS_PATH was set in the Colab setup cell
CHECKPOINT_PATH = Path(CHECKPOINT_DIR) / CHECKPOINT_NAME
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Cannot find {CHECKPOINT_PATH}. Run the training cell first.")
VAL_PAIRS_PATH = Path(VAL_PAIRS_PATH)

checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
state = checkpoint["model_state_dict"]

# 2. Reconstruct and load the selected model
class FaceModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.backbone = timm.create_model(
            "mobilenetv2_100", pretrained=False,
            num_classes=0, global_pool="avg"
        )

        if MODEL_TYPE == "classification":
            self.classifier = nn.Linear(
                self.backbone.num_features, num_classes
            )
        else:
            self.embedding_layer = nn.Linear(
                self.backbone.num_features, 512
            )
            self.bn = nn.BatchNorm1d(512)
            self.head = nn.Linear(512, num_classes, bias=False)

    def forward(self, x):
        features = self.backbone(x)

        if MODEL_TYPE == "classification":
            return self.classifier(features)

        embeddings = self.bn(self.embedding_layer(features))
        return F.normalize(embeddings, p=2, dim=1)


num_classes = state[
    "classifier.weight" if MODEL_TYPE == "classification"
    else "head.weight"
].shape[0]

model = FaceModel(num_classes).to(device)
model.load_state_dict(state)
model.eval()

print(f"Loaded {MODEL_TYPE} checkpoint: {CHECKPOINT_PATH}")

# 3. Load verification pairs
pairs = pd.read_csv(
    VAL_PAIRS_PATH, sep=r"\s+", header=None,
    names=["img1", "img2", "label"]
)

def resolve_path(path):
    path = Path(path).expanduser()
    for p in [path, CURRENT_DIR / path,
              VAL_PAIRS_PATH.parent / path,
              VAL_PAIRS_PATH.parent.parent / path]:
        if p.is_file():
            return p.resolve()
    raise FileNotFoundError(path)

class VerificationDataset(Dataset):
    def __len__(self):
        return len(pairs)

    def __getitem__(self, idx):
        row = pairs.iloc[idx]

        with Image.open(resolve_path(row.img1)) as img:
            img1 = transform(img.convert("RGB"))

        with Image.open(resolve_path(row.img2)) as img:
            img2 = transform(img.convert("RGB"))

        return img1, img2, int(row.label)

transform = transforms.Compose([
    transforms.Resize((112, 112)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3)
])

val_loader = DataLoader(
    VerificationDataset(), batch_size=128,
    shuffle=False, num_workers=NUM_WORKERS
)

# 4. Generate embeddings and calculate similarities
similarities, labels = [], []

with torch.no_grad():
    for img1, img2, target in tqdm(val_loader, desc="Validating"):
        img1, img2 = img1.to(device), img2.to(device)

        if MODEL_TYPE == "classification":
            emb1 = F.normalize(model.backbone(img1), dim=1)
            emb2 = F.normalize(model.backbone(img2), dim=1)
        else:
            emb1, emb2 = model(img1), model(img2)

        similarities.extend((emb1 * emb2).sum(dim=1).cpu().numpy())
        labels.extend(target.numpy())

similarities = np.array(similarities)
labels = np.array(labels)

# 5. Evaluation metrics
auc = roc_auc_score(labels, similarities)

fixed_threshold = 0.5
fixed_acc = accuracy_score(labels, similarities >= fixed_threshold)

fpr, tpr, thresholds = roc_curve(labels, similarities)
best_threshold = thresholds[np.argmax(tpr - fpr)]
best_acc = accuracy_score(labels, similarities >= best_threshold)

# 6. Results
print(f"\n========== {MODEL_TYPE.upper()} EVALUATION ==========")
print(f"Verification pairs: {len(labels)}")
print(f"ROC-AUC: {auc:.4f}")
print(f"Accuracy (threshold 0.50): {fixed_acc:.4f}")
print(f"Best validation threshold: {best_threshold:.4f}")
print(f"Accuracy (best threshold): {best_acc:.4f}")
print("==========================================")


Loaded metric_learning checkpoint: /content/drive/MyDrive/face_recognition/models/mobilenetv2_metric_learning_checkpoint.pth


Validating:   0%|          | 0/69 [00:00<?, ?it/s]


========== METRIC_LEARNING EVALUATION ==========
Verification pairs: 8805
ROC-AUC: 0.7755
Accuracy (threshold 0.50): 0.7110
Best validation threshold: 0.4567
Accuracy (best threshold): 0.7230


### Extract an embedding from a custom image

In [8]:
with open(VAL_PAIRS_PATH, "r", encoding="utf-8") as f:
    for i in range(5):
        line = f.readline()
        print(f"Line {i + 1}: {repr(line)}")
        print("Values:", line.split())

Line 1: 'verification_data/00041961.jpg verification_data/00044353.jpg 0\n'
Values: ['verification_data/00041961.jpg', 'verification_data/00044353.jpg', '0']
Line 2: 'verification_data/00007133.jpg verification_data/00060449.jpg 1\n'
Values: ['verification_data/00007133.jpg', 'verification_data/00060449.jpg', '1']
Line 3: 'verification_data/00041961.jpg verification_data/00020166.jpg 0\n'
Values: ['verification_data/00041961.jpg', 'verification_data/00020166.jpg', '0']
Line 4: 'verification_data/00013102.jpg verification_data/00055525.jpg 1\n'
Values: ['verification_data/00013102.jpg', 'verification_data/00055525.jpg', '1']
Line 5: 'verification_data/00002921.jpg verification_data/00041331.jpg 0\n'
Values: ['verification_data/00002921.jpg', 'verification_data/00041331.jpg', '0']
